# GLM-5 / Kimi Distillation: Eval & Publish

**Runs on Colab Pro** (A100/V100) for fast local GPU inference.

Uses [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) for all benchmarks —
the industry standard used by HuggingFace Open LLM Leaderboard. No custom extraction code.

## Benchmarks

| Benchmark | Shots | Method | Task name | Notes |
|-----------|-------|--------|-----------|-------|
| GSM8K | 8 (CoT) | Generative | `gsm8k_cot` | Standard CoT, regex extraction |
| MATH | 4 | Generative | `minerva_math` | Uses `\boxed{}` extraction |
| ARC-Challenge | 25 | Log-likelihood | `arc_challenge` | No generation needed |
| GPQA Diamond | 0 | Log-likelihood | `gpqa_diamond` | Jackrong comparison |
| MMLU-Pro | 5 | Log-likelihood | `mmlu_pro` | Knowledge + reasoning |

## Models

1. Base Qwen3.5-4B (baseline)
2. Distilled GLM-5 (SFT)
3. Distilled Kimi (SFT)
4. Distilled Combined (SFT)
5. GRPO GLM-5 (SFT + GRPO) — if available
6. GRPO Kimi (SFT + GRPO) — if available
7. GRPO Combined (SFT + GRPO) — if available
8. Llama-3.2-3B (reference)
9. Qwen3-8B (reference)

gpt-oss-20b too large for single GPU — use published numbers.

In [ ]:
# Step 0: Install everything
!pip install unsloth lm-eval tinker transformers datasets huggingface_hub

from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json
DRIVE_DIR = '/content/drive/MyDrive/distillreasoning'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/eval_results', exist_ok=True)
print(f'Drive dir: {DRIVE_DIR}')

In [ ]:
# Step 1: Download LoRA weights from Tinker and merge
import tinker, requests, zipfile
from unsloth import FastLanguageModel

os.environ['TINKER_API_KEY'] = userdata.get('TINKER_API_KEY')
service = tinker.ServiceClient()
rest = service.create_rest_client()

# Update these paths after GRPO training
CHECKPOINTS = {
    'distilled-glm5': 'tinker://0fbca836-2aae-5500-b28d-93c2a46a328b:train:0/sampler_weights/qwen35-4b-glm5-final',
    'distilled-kimi': 'tinker://f5795e66-71e4-5cf4-9ebe-1cc14c27aa6e:train:0/sampler_weights/qwen35-4b-kimi-final',
    'distilled-combined': 'tinker://41e7bd9e-e49f-5f13-a5ff-f4339faab448:train:0/sampler_weights/qwen35-4b-combined-final',
    # Add GRPO checkpoints here after GRPO training:
    # 'grpo-glm5': 'tinker://...',
    # 'grpo-kimi': 'tinker://...',
    # 'grpo-combined': 'tinker://...',
}

BASE_MODEL = 'Qwen/Qwen3.5-4B'

for name, tinker_path in CHECKPOINTS.items():
    merged_dir = f'{DRIVE_DIR}/merged_{name}'
    if os.path.exists(merged_dir):
        print(f'{name}: already merged at {merged_dir}, skipping')
        continue
    
    print(f'\nDownloading {name}...')
    url_resp = rest.get_checkpoint_archive_url_from_tinker_path(tinker_path).result()
    local_zip = f'/tmp/lora_{name}.zip'
    r = requests.get(url_resp.url, stream=True)
    with open(local_zip, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    
    lora_dir = f'/tmp/lora_{name}'
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(lora_dir)
    
    print(f'  Merging with {BASE_MODEL}...')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=lora_dir, max_seq_length=4096, dtype=None, load_in_4bit=False,
    )
    model.save_pretrained_merged(merged_dir, tokenizer, save_method='merged_16bit')
    del model, tokenizer
    import torch; torch.cuda.empty_cache()
    print(f'  Saved: {merged_dir}')

print('\nAll models ready!')

In [ ]:
# Step 2: Run lm-evaluation-harness on all models
#
# This is the STANDARD eval used by HuggingFace Open LLM Leaderboard.
# No custom extraction code — lm-eval handles everything.
#
# For chat/instruct models: --apply_chat_template --fewshot_as_multiturn
# For base models: no chat template flags
#
# H100 (80GB) can run gpt-oss-20b in bfloat16 (~40GB)

# Benchmarks to run (standard settings from Open LLM Leaderboard v2)
BENCHMARKS = {
    'gsm8k_cot':      {'num_fewshot': 8,  'type': 'generative'},
    'minerva_math':    {'num_fewshot': 4,  'type': 'generative'},   # Uses \boxed{} extraction
    'arc_challenge':   {'num_fewshot': 25, 'type': 'log-likelihood'},
    'gpqa_diamond':    {'num_fewshot': 0,  'type': 'log-likelihood'},
    'mmlu_pro':        {'num_fewshot': 5,  'type': 'log-likelihood'},
}

# Models to evaluate
MODELS = [
    # (name, model_path, is_chat_model)
    ('base-4b', 'Qwen/Qwen3.5-4B', True),
    ('llama-3b', 'meta-llama/Llama-3.2-3B', False),
    ('qwen3-8b', 'Qwen/Qwen3-8B', False),
    ('gpt-oss-20b', 'openai/gpt-oss-20b', True),  # Fits on H100 80GB in bf16
]

# Add our merged distilled models
for name in CHECKPOINTS:
    merged_dir = f'{DRIVE_DIR}/merged_{name}'
    if os.path.exists(merged_dir):
        MODELS.append((name, merged_dir, True))

print(f'Models to evaluate: {len(MODELS)}')
for name, path, chat in MODELS:
    print(f'  {name:25s} chat={chat}')

In [ ]:
# Step 2b: Run all evals
# This cell runs lm-eval for each model × benchmark combination.
# Results are saved to Drive so they survive runtime disconnects.

import subprocess

all_tasks = ','.join(BENCHMARKS.keys())

for name, model_path, is_chat in MODELS:
    result_dir = f'{DRIVE_DIR}/eval_results/{name}'
    
    # Check if already completed
    result_file = f'{result_dir}/results.json'
    if os.path.exists(result_file):
        print(f'{name}: already evaluated, skipping')
        continue
    
    print(f'\n{"="*60}')
    print(f'Evaluating: {name}')
    print(f'{"="*60}')
    
    cmd = [
        'lm-eval',
        '--model', 'hf',
        '--model_args', f'pretrained={model_path},dtype=bfloat16,trust_remote_code=True',
        '--tasks', all_tasks,
        '--batch_size', 'auto',
        '--output_path', result_dir,
        '--log_samples',  # Save individual predictions for inspection
    ]
    
    if is_chat:
        cmd.extend(['--apply_chat_template', '--fewshot_as_multiturn'])
    
    print(f'  Command: {" ".join(cmd)}')
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-2000:] if result.stdout else 'No output')
    if result.returncode != 0:
        print(f'  ERROR: {result.stderr[-1000:]}')
    else:
        print(f'  ✅ Results saved to {result_dir}')

print('\nAll evaluations complete!')

In [ ]:
# Step 3: Collect and display results
import json, glob

# Parse lm-eval output files
all_results = {}

for name, _, _ in MODELS:
    result_files = glob.glob(f'{DRIVE_DIR}/eval_results/{name}/**/results.json', recursive=True)
    if not result_files:
        continue
    
    with open(result_files[0]) as f:
        data = json.load(f)
    
    results = {}
    for task_name, task_data in data.get('results', {}).items():
        # lm-eval stores metrics like 'acc,none' or 'exact_match,strict-match'
        acc = task_data.get('acc,none', task_data.get('acc_norm,none', 
              task_data.get('exact_match,strict-match', task_data.get('exact_match,flexible-extract', None))))
        if acc is not None:
            results[task_name] = round(acc * 100, 1)
    
    all_results[name] = results

# Print comparison table
benchmarks = ['gsm8k_cot', 'minerva_math', 'arc_challenge', 'gpqa_diamond', 'mmlu_pro']
labels = ['GSM8K 8-CoT', 'MATH', 'ARC 25-shot', 'GPQA 0-shot', 'MMLU-Pro']

print(f'{"Model":25s}', end='')
for label in labels:
    print(f'{label:>12s}', end='')
print()
print('-' * (25 + 12 * len(labels)))

for name, _, _ in MODELS:
    if name not in all_results:
        continue
    print(f'{name:25s}', end='')
    for bench in benchmarks:
        score = all_results[name].get(bench, '—')
        print(f'{str(score) + "%" if isinstance(score, float) else score:>12s}', end='')
    print()

# Add published reference for gpt-oss-20b
print(f'{"gpt-oss-20b (published)":25s}', end='')
for val in ['68.9%*', '—', '72.5%', '71.5%', '—']:
    print(f'{val:>12s}', end='')
print()
print('\n* Published numbers use different eval settings')

# Save combined results
with open(f'{DRIVE_DIR}/eval_results/comparison.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nSaved to {DRIVE_DIR}/eval_results/comparison.json')

In [ ]:
# Step 4: Qualitative trick questions (manual, not lm-eval)
from unsloth import FastLanguageModel
import torch

TRICK_QUESTIONS = [
    ('A farmer has 17 sheep. All but 9 die. How many sheep does the farmer have left?', '9'),
    ('If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?', '5 minutes'),
    ('A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?', '$0.05'),
    ('A store offers a 20% discount on a jacket that costs $80. Then they add 8% sales tax. What is the final price?', '$69.12'),
    ('If all roses are flowers and some flowers fade quickly, can we conclude that some roses fade quickly?', 'No'),
]

SYSTEM_MSG = 'You are a helpful reasoning assistant. Think through problems step by step before answering.'

trick_results = {}
for model_name, model_path, is_chat in MODELS:
    print(f'\nTrick questions: {model_name}')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path, max_seq_length=4096, dtype=None, load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    
    trick_results[model_name] = []
    for q, expected in TRICK_QUESTIONS:
        messages = [{'role': 'system', 'content': SYSTEM_MSG}, {'role': 'user', 'content': q}]
        inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to('cuda')
        with torch.no_grad():
            outputs = model.generate(inputs, max_new_tokens=2048, temperature=0.6, top_p=0.95, do_sample=True)
        response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
        trick_results[model_name].append({'problem': q, 'expected': expected, 'response': response[:500]})
    
    del model, tokenizer
    torch.cuda.empty_cache()

# Print side by side
for i, (q, expected) in enumerate(TRICK_QUESTIONS):
    print(f'\n{"="*70}')
    print(f'Q: {q}')
    print(f'Expected: {expected}')
    for name in trick_results:
        resp = trick_results[name][i]['response'][:300]
        print(f'\n  [{name}]')
        print(f'  {resp}')

with open(f'{DRIVE_DIR}/eval_results/trick_questions.json', 'w') as f:
    json.dump(trick_results, f, indent=2)

In [ ]:
# Step 5: Inspect individual predictions (for devlog examples)
# lm-eval with --log_samples saves every prediction

import glob, json

# Look at a few GSM8K predictions from distilled-kimi
sample_files = glob.glob(f'{DRIVE_DIR}/eval_results/distilled-kimi/**/samples_gsm8k*.jsonl', recursive=True)
if sample_files:
    with open(sample_files[0]) as f:
        samples = [json.loads(l) for l in f][:5]
    
    for s in samples:
        print(f'Problem: {s.get("doc", {}).get("question", "")[:100]}...')
        print(f'Expected: {s.get("target", "?")}')
        print(f'Model: {str(s.get("resps", [[""]])[0][0])[:200]}...')
        print(f'Correct: {s.get("exact_match", "?")}')
        print()
else:
    print('No sample logs found. Make sure --log_samples was used.')

In [ ]:
# Step 6: Push best model to HuggingFace
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

# Pick best model based on GSM8K CoT score
distilled_results = {k: v for k, v in all_results.items() if 'distilled' in k or 'grpo' in k}
best_name = max(distilled_results, key=lambda k: distilled_results[k].get('gsm8k_cot', 0))
print(f'Best model: {best_name} (GSM8K: {distilled_results[best_name].get("gsm8k_cot", "?")}%)')

best_dir = f'{DRIVE_DIR}/merged_{best_name}'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=best_dir, max_seq_length=4096, dtype=None, load_in_4bit=False,
)

model.push_to_hub_merged(
    'bmeyer2025/qwen3.5-4b-glm5-reasoning-distilled',
    tokenizer,
    save_method='merged_16bit',
    token=userdata.get('HF_TOKEN'),
)
print('Pushed merged model to HuggingFace!')

In [ ]:
# Step 7: Export GGUF
model.push_to_hub_gguf(
    'bmeyer2025/qwen3.5-4b-glm5-reasoning-distilled',
    tokenizer,
    quantization_method=['q4_k_m', 'q8_0'],
    token=userdata.get('HF_TOKEN'),
)
print('GGUF files pushed!')
print('Run locally: ollama run hf.co/bmeyer2025/qwen3.5-4b-glm5-reasoning-distilled:Q4_K_M')

## Notes

**Why lm-evaluation-harness?**
- Industry standard (used by HuggingFace Open LLM Leaderboard)
- Handles answer extraction correctly per benchmark
- ARC, GPQA, MMLU-Pro use log-likelihood scoring (no generation/extraction needed)
- GSM8K uses regex: `The answer is (\-?[0-9\.\,]*[0-9]+)`
- MATH (minerva) uses `\boxed{}` extraction with LaTeX normalization
- `--log_samples` saves every prediction for manual inspection

**Why not custom eval code?**
We tried custom extraction first and it broke badly:
- MATH: 0% on base models (only checked `\boxed{}`, base models don't use it)
- ARC: 100% on base models (regex too generous, matched random capitals)
- GSM8K worked but only because number extraction is simpler

Using lm-eval means our results are directly comparable to every model card on HuggingFace.

**Published reference (gpt-oss-20b):**
- GSM8K: 68.9%, ARC: 72.5%, MMLU: 85.3%, GPQA: 71.5%
- Too large for single A100, using published numbers instead